In [1]:
import os
import torch
import matplotlib as plt
import numpy as np
import pandas as pd
import scipy.io as wavfile
from python_speech_features import mfcc, logfbank
import librosa
import IPython
from IPython import display as ipd
from tqdm import tqdm
import soundfile as sf
from collections import defaultdict
import copy
from src.audio_processing import audio_file_to_logmels, get_signal_centered_chunks_from_array
import src.precompute as pc
from src.dataset import BirdChunkDataset
from src.model import BirdResNet
from src.templates import build_species_templates,compute_overlap,filter_secondary_labels_for_chunk
from src.labels import clean_secondary_labels, build_all_labels, build_class_list
from src.splits import build_file_level_df, create_splits
from src.labels import encode_labels
import pickle
from src.config import (
    SR,
    N_FFT,
    HOP_LENGTH,
    FMIN,
    FMAX,
    CHUNK_CONFIG
)
from src.dataset import BirdChunkDataset
from torch.utils.data import DataLoader

# run experiments, call functions, visualize results

## Loading Data and Pre-Processing

In [2]:
train = pd.read_csv('train.csv')

In [3]:
train_soundscapes = pd.read_csv('train_soundscapes_labels.csv')

In [4]:
train.shape

(35549, 15)

In [5]:
train1 = train.copy()

In [6]:
train2 = train1.copy()

In [7]:
#adding the full path to fetch the audio
base_path = "/Users/sauravbellam/Desktop/Projects/audio classification/birdclef-2026 2/train_audio"

train2['full_path'] = train2['filename'].apply(lambda x: os.path.join(base_path, x))
train_soundscapes['full_path'] = train_soundscapes['filename'].apply(lambda x: os.path.join(base_path, x))

In [8]:
train3 = train2.copy()
train_soundscapes1 = train_soundscapes.copy()

In [9]:
train4 = train3.drop(columns =['collection', 'author', 'license', 'url', 'inat_taxon_id', 'common_name', 'type'])

## Master Dataset

In [10]:
rate5 = train4[train4['rating'] == 5]

In [11]:
import ast

def parse_secondary(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return []


In [12]:
secondary = train4[train4['secondary_labels'].apply(lambda x: len(x) > 0)]

In [13]:
secondary_low = secondary[secondary['rating'] < 3]

In [14]:
clean_df = train4[train4["rating"] == 5].copy()
clean_df["phase_group"] = "clean"

semi_df = secondary[(secondary["rating"] >= 3) & (secondary["rating"] < 5)].copy()
semi_df["phase_group"] = "semi"

messy_df = pd.concat(
    [train_soundscapes, secondary_low],
    axis=0
).reset_index(drop=True).copy()
messy_df["phase_group"] = "messy"

master_df = pd.concat(
    [clean_df, semi_df, messy_df],
    axis=0,
    ignore_index=True
).copy()

In [15]:
master_df = clean_secondary_labels(master_df)
master_df = build_all_labels(master_df)

class_list, label_to_idx = build_class_list(master_df)
num_classes = len(class_list)
def encode_fn(labels):
    return encode_labels(labels, label_to_idx, num_classes)

In [16]:
master_df["file_id"] = master_df["full_path"].apply(
    lambda x: os.path.splitext(os.path.basename(x))[0]
)

In [17]:
with open("templates.pkl", "rb") as f:
    templates = pickle.load(f)

## Create splits

In [18]:
# 1. Build file-level dataset
file_level_df = build_file_level_df(master_df)

# 2. Create splits
file_level_df = create_splits(file_level_df)

# 3. Encode targets
file_level_df["target"] = file_level_df["all_labels"].apply(
    lambda labels: encode_labels(labels, label_to_idx, num_classes)
)

# 4. Save (optional but recommended)
file_level_df.to_pickle("splits/file_level_df.pkl")

DEBUG: trainval size = 30272
DEBUG: cv_fold distribution:
cv_fold
1    6055
0    6055
2    6054
3    6054
4    6054
Name: count, dtype: int64


## precompute

In [19]:
#Helper Function
def make_subset_file_df(file_df, frac=0.2, random_state=42):
    return (
        file_df.groupby("phase_group", group_keys=False)
        .apply(lambda x: x.sample(frac=frac, random_state=random_state))
        .reset_index(drop=True)
    )

In [20]:
#precompute all folds using parralel cpu
for fold in range(5):
    pc.precompute_fold(
        fold=fold,
        file_level_df=file_level_df,
        master_df=master_df,
        templates=templates,
        encode_labels=encode_fn
    )


=== Processing Fold 0 ===
Train rows: 25192 | Val rows: 6354


  0%|          | 0/25192 [00:00<?, ?it/s]

ValueError: n_jobs == 0 in Parallel has no meaning

## Inital Chunk Analysis 
### Trying to optimize parameters without too much computational cost. Later we will use optimzer to find best global and local params

In [ ]:
samples = precompute_chunk_cache(
    df=subset_rows,
    output_dir="debug_chunks",
    templates=templates,
    encode_labels=encode_fn,
    sr=16000,
    duration=5,
    only_strongest=False,
    fallback_to_regular_split=True,       
    band_peak_rel_db=6.0,
    threshold_db = 6.0,
    min_event_duration = 0.18,
    merge_gap = 0.8           
)

In [ ]:
print(len(samples))
#basic chunk analysis
df_samples = pd.DataFrame(samples)

# Avg chunks per file
chunks_per_file = df_samples.groupby("source_path").size()

print("Avg chunks per file:", chunks_per_file.mean())
print("Min:", chunks_per_file.min(), "Max:", chunks_per_file.max())

# Fallback rate
print("Fallback rate:", df_samples["used_fallback"].mean())

In [ ]:
#the max chunk files
df_samples = pd.DataFrame(samples)

chunks_per_file = df_samples.groupby("source_path").size()

top_files = chunks_per_file.sort_values(ascending=False).head(10)

print(top_files)

In [ ]:
#looks normal just long length files
for path in top_files.index[:5]:
    y, sr = librosa.load(path, sr=16000)
    duration = len(y) / sr
    print(f"{path} → {duration:.2f} sec")

# Optuna

In [47]:
#1 to 5 folds model
import optuna
import torch
import numpy as np
from torch.utils.data import DataLoader

device = torch.device("mps" if torch.cuda.is_available() else "cpu")

def objective(trial, use_full_cv=False):

    # -----------------------------
    # 1. Hyperparameters
    # -----------------------------
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    use_scheduler = trial.suggest_categorical("scheduler", [True, False])

    num_epochs = 3  # keep small for tuning

    # -----------------------------
    # 2. Choose folds
    # -----------------------------
    folds = range(5) if use_full_cv else [0]

    fold_scores = []

    for fold in folds:

        # -----------------------------
        # 3. Load precomputed data
        # -----------------------------
        train_samples = load_samples_for_fold(fold, "train")
        val_samples   = load_samples_for_fold(fold, "val")

        train_dataset = BirdChunkDataset(train_samples, augment=True)
        val_dataset   = BirdChunkDataset(val_samples, augment=False)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        # -----------------------------
        # 4. Compute pos_weight (file-level)
        # -----------------------------
        train_files = file_level_df[
            (file_level_df["split_role"] == "trainval") &
            (file_level_df["cv_fold"] != fold)
        ]

        Y = np.stack(train_files["target"].values)
        pos_counts = Y.sum(axis=0)
        neg_counts = len(Y) - pos_counts

        pos_weight = neg_counts / (pos_counts + 1e-6)
        pos_weight = np.clip(pos_weight, 1, 20)

        pos_weight = torch.tensor(pos_weight, dtype=torch.float32).to(device)

        criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        # -----------------------------
        # 5. Model
        # -----------------------------
        model = BirdResNet(num_classes).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        if use_scheduler:
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=num_epochs
            )
        else:
            scheduler = None

        # -----------------------------
        # 6. Training
        # -----------------------------
        for epoch in range(num_epochs):
            model.train()

            for X, y in train_loader:
                X, y = X.to(device), y.to(device)

                optimizer.zero_grad()
                logits = model(X)
                loss = criterion(logits, y)
                loss.backward()
                optimizer.step()

            if scheduler:
                scheduler.step()

        # -----------------------------
        # 7. Validation (ROC-AUC proxy)
        # -----------------------------
        model.eval()
        preds = []
        targets = []

        with torch.no_grad():
            for X, y in val_loader:
                X = X.to(device)
                logits = model(X)
                probs = torch.sigmoid(logits).cpu().numpy()

                preds.append(probs)
                targets.append(y.numpy())

        preds = np.vstack(preds)
        targets = np.vstack(targets)

        # simple metric (replace later with ROC-AUC)
        score = ((preds > 0.5) == targets).mean()

        fold_scores.append(score)

    return np.mean(fold_scores)

In [48]:
#phase 1
study = optuna.create_study(direction="maximize")

study.optimize(lambda trial: objective(trial, use_full_cv=False), n_trials=15)

[I 2026-04-16 06:05:51,431] A new study created in memory with name: no-name-d06ba709-e213-44b2-b19e-f10d31cceec7
[W 2026-04-16 06:05:51,447] Trial 0 failed with parameters: {'lr': 0.002158896700818624, 'batch_size': 32, 'scheduler': False} because of the following error: ValueError('num_samples should be a positive integer value, but got num_samples=0').
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/0b/chkmxb0j79nfjfnjq2zw2lnc0000gn/T/ipykernel_9784/740576294.py", line 4, in <lambda>
    study.optimize(lambda trial: objective(trial, use_full_cv=False), n_trials=15)
                                 ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/0b/chkmxb0j79nfjfnjq2zw2lnc0000gn/T/ipykernel_9784/403873581.py", line 38, in objective
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  File "/opt/anacon

ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
#phase 2
study.optimize(lambda trial: objective(trial, use_full_cv=True), n_trials=10)

In [ ]:
#phase 3


## Training

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for fold in range(5):

    print(f"\n========== FOLD {fold} ==========")

    # -----------------------------
    # 1. Split file-level
    # -----------------------------
    train_files = file_level_df[
        (file_level_df["split_role"] == "trainval") &
        (file_level_df["cv_fold"] != fold)
    ]

    val_files = file_level_df[
        (file_level_df["split_role"] == "trainval") &
        (file_level_df["cv_fold"] == fold)
    ]

    # -----------------------------
    # 2. Convert to row-level
    # -----------------------------
    train_df = get_rows_for_files(master_df, train_files)
    val_df   = get_rows_for_files(master_df, val_files)

    print("Train rows:", len(train_df), "| Val rows:", len(val_df))

    # -----------------------------
    # 3. Compute pos_weight (CRITICAL)
    # -----------------------------
    Y = np.stack(train_files["target"].values)  # ← FILE LEVEL

    pos_counts = Y.sum(axis=0)
    neg_counts = len(Y) - pos_counts

    pos_weight = neg_counts / (pos_counts + 1e-6)
    pos_weight = np.clip(pos_weight, 1, 20)

    pos_weight = torch.tensor(pos_weight, dtype=torch.float32).to(device)

    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    # -----------------------------
    # 4. Load precomputed samples
    # -----------------------------
    train_samples = load_samples_for_fold(fold, split="train")
    val_samples   = load_samples_for_fold(fold, split="val")

    # -----------------------------
    # 5. Dataset
    # -----------------------------
    train_dataset = BirdChunkDataset(
        train_samples,
        augment=True
    )

    val_dataset = BirdChunkDataset(
        val_samples,
        augment=False
    )

    # -----------------------------
    # 6. DataLoader
    # -----------------------------
    train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True,
        num_workers=2
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=2
    )

    # -----------------------------
    # 7. Model
    # -----------------------------
    model = BirdResNet(num_classes).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # -----------------------------
    # 8. Training loop
    # -----------------------------
    for epoch in range(5):

        model.train()
        total_loss = 0

        for X, y in train_loader:
            X, y = X.to(device), y.to(device)

            optimizer.zero_grad()

            logits = model(X)
            loss = criterion(logits, y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch} Train Loss: {total_loss:.4f}")

        # -----------------------------
        # Validation
        # -----------------------------
        model.eval()
        val_loss = 0

        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                logits = model(X)
                loss = criterion(logits, y)
                val_loss += loss.item()

        print(f"Epoch {epoch} Val Loss: {val_loss:.4f}")

### weighting

In [57]:
# 1. stack targets
Y = np.stack(train4['target'].values)

# 2. counts
pos_counts = Y.sum(axis=0)
neg_counts = len(Y) - pos_counts

# 3. compute weights
pos_weight = neg_counts / (pos_counts + 1e-6)

# CLIP HERE - account for instability
pos_weight = np.clip(pos_weight, 1, 20)

# 5. convert to tensor
pos_weight = torch.tensor(pos_weight, dtype=torch.float32)

# 6. move to device (if using GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pos_weight = pos_weight.to(device)

# 7. define loss
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

## Training Loop

In [58]:
from torch.utils.data import DataLoader

In [59]:
phase1_samples = precompute_dataset(phase1_train, BASE_PATH, "spec_phase1")
phase2_samples = precompute_dataset(phase2_train, BASE_PATH, "spec_phase2")
phase3_samples = precompute_dataset(phase3_train, BASE_PATH, "spec_phase3")

  1%|▍                                        | 51/4791 [00:04<07:32, 10.49it/s]


KeyboardInterrupt: 

In [60]:
import os
import pandas as pd

def load_samples_from_dir(df, spec_dir):
    samples = []

    spec_files = os.listdir(spec_dir)

    for file in spec_files:
        if not file.endswith(".npy"):
            continue

        # extract original row index
        idx = int(file.split("_")[0])

        spec_path = os.path.join(spec_dir, file)

        samples.append({
            "spec_path": spec_path,
            "target": df.iloc[idx]["target"]
        })

    return samples

In [61]:
phase1_samples = load_samples_from_dir(phase1_train, "spec_phase1")
phase2_samples = load_samples_from_dir(phase2_train, "spec_phase2")
phase3_samples = load_samples_from_dir(phase3_train, "spec_phase3")

In [62]:
val_samples = load_samples_from_dir(val_df, "spec_val")

In [63]:
train_loader = DataLoader(
    BirdDataset(phase2_samples, augment=True),
    batch_size=32,
    shuffle=True,
    num_workers=0
)

print("Testing train_loader...")

for x, y in train_loader:
    print("✅ First batch loaded")

    print("Input shape:", x.shape)
    print("Target shape:", y.shape)

    print("Input dtype:", x.dtype)
    print("Target dtype:", y.dtype)

    break

Testing train_loader...
✅ First batch loaded
Input shape: torch.Size([32, 1, 128, 157])
Target shape: torch.Size([32, 234])
Input dtype: torch.float32
Target dtype: torch.float32


In [81]:
num_classes = 234

In [79]:
import optuna
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

def objective(trial):
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    scheduler_type = trial.suggest_categorical("scheduler", ["none", "cosine"])

    num_epochs = 5

    model = BirdResNet(num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

    if scheduler_type == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=num_epochs
        )
    else:
        scheduler = None

    train_loader = DataLoader(
        BirdDataset(phase3_samples, augment=True),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    val_loader = DataLoader(
        BirdDataset(val_samples, augment=False),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    print(f"Trial {trial.number} started | lr={lr:.6f} bs={batch_size} sched={scheduler_type}")

    for epoch in range(num_epochs):
        print(f"Trial {trial.number} - Epoch {epoch+1}/{num_epochs}")

        model.train()
        train_loss = 0.0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            print(" batch loaded")
            
            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        if scheduler is not None:
            scheduler.step()

        avg_train_loss = train_loss / len(train_loader)
        print(f"Trial {trial.number} - Avg train loss: {avg_train_loss:.4f}")

    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)

            outputs = model(x)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()

            all_preds.append(preds.cpu())
            all_targets.append(y.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()

    f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    print(f"Trial {trial.number} finished | F1={f1:.4f}")

    return f1

In [82]:
study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=5, show_progress_bar=True)

print("Best params:", study.best_params)

[I 2026-04-15 16:12:16,233] A new study created in memory with name: no-name-71ea2e6f-f76c-4b54-b8f9-be0c20c61686


  0%|          | 0/5 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Trial 0 started | lr=0.008419 bs=64 sched=none
Trial 0 - Epoch 1/5
 batch loaded
 batch loaded
 batch loaded
 batch loaded
 batch loaded
 batch loaded
 batch loaded
 batch loaded
 batch loaded
 batch loaded
 batch loaded
 batch loaded
 batch loaded
 batch loaded
 batch loaded
 batch loaded
[W 2026-04-15 16:12:41,196] Trial 0 failed with parameters: {'lr': 0.008418754783433905, 'batch_size': 64, 'scheduler': 'none'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/0b/chkmxb0j79nfjfnjq2zw2lnc0000gn/T/ipykernel_6376/634333291.py", line 54, in objective
    loss.backward()
    ~~~~~~~~~~~~~^^
  File "/opt/anaconda3/lib/python3.13/site-packages/torch/_tensor.py", line 630, in backward
    torch.autograd.backward(
    ~~~~~~~~~~~~~~~~~~~~~~~^
        self, gradient, retain_graph, create_graph, inp

KeyboardInterrupt: 

In [64]:
def get_train_loader(epoch, batch_size=32):

    if epoch < 5:
        dataset = BirdDataset(phase1_samples, augment=True)

    elif epoch < 10:
        dataset = BirdDataset(phase2_samples, augment=True)

    else:
        dataset = BirdDataset(phase3_samples, augment=True)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True
    )

In [65]:
num_classes = len(train4.iloc[0]['target'])

In [66]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BirdResNet(num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# pos_weight already computed
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

num_epochs = 15

for epoch in range(num_epochs):

    train_loader = get_train_loader(epoch)

    model.train()
    train_loss = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(x)
        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # VALIDATION
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)

            outputs = model(x)
            loss = criterion(outputs, y)

            val_loss += loss.item()

    print(f"Epoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")

/opt/anaconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


NameError: name 'phase1_samples' is not defined

In [ ]:
probs = torch.sigmoid(outputs)

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=2, factor=0.5
)

In [ ]:
scheduler.step(val_loss)

In [ ]:
best_val = float("inf")
patience = 3
counter = 0

if val_loss < best_val:
    best_val = val_loss
    counter = 0
    torch.save(model.state_dict(), "best_model.pt")
else:
    counter += 1
    if counter >= patience:
        print("Early stopping")
        break